# 06 — Model Training

## Objective

Train baseline machine-learning models to predict 30-day hospital readmission.

### Target
- `readmitted_30`
- 0 = not readmitted within 30 days
- 1 = readmitted within 30 days

### Data Split
- Patient-level 80/20 split
- Random state: 42
- No patient overlap between train and test
- No encounter overlap between train and test

### Leakage Prevention
All preprocessing transformations must be fitted only on the training data and then applied to the test data.

### Scope
This notebook focuses on baseline model training.
Model evaluation, SHAP explainability, and the prediction dashboard will be handled in later stages.

In [27]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

## 2. Load Data and Recreate Patient-Level Train/Test Split

The dataset is split at the patient level to prevent encounters from the same patient appearing in both training and test sets.

The split uses:
- 80% of unique patients for training
- 20% of unique patients for testing
- `random_state = 42`

In [28]:
DATA_PATH = "../data/raw/diabetic_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)


Dataset shape: (101766, 50)


In [29]:
TARGET = "readmitted_30"
PATIENT_ID = "patient_nbr"

# Create 30-day readmission target
df[TARGET] = (df["readmitted"] == "<30").astype(int)

print("Target distribution:")
print(df[TARGET].value_counts())

print("\nTarget proportions:")
print(df[TARGET].value_counts(normalize=True))

Target distribution:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64

Target proportions:
readmitted_30
0    0.888401
1    0.111599
Name: proportion, dtype: float64


In [30]:
# Load the approved patient-level split created in Stage 5
TRAIN_PATIENTS_PATH = "../data/processed/split/train_patients.csv"
TEST_PATIENTS_PATH = "../data/processed/split/test_patients.csv"

train_patients = pd.read_csv(TRAIN_PATIENTS_PATH)[PATIENT_ID]
test_patients = pd.read_csv(TEST_PATIENTS_PATH)[PATIENT_ID]

print("Training patients:", train_patients.nunique())
print("Test patients:", test_patients.nunique())

# Verify that the approved patient split has no overlap
patient_overlap = set(train_patients).intersection(set(test_patients))

print("Patients in both split files:", len(patient_overlap))

if len(patient_overlap) == 0:
    print("PASS: Approved split has no patient overlap.")
else:
    print("WARNING: Patient overlap detected!")

Training patients: 57214
Test patients: 14304
Patients in both split files: 0
PASS: Approved split has no patient overlap.


In [31]:
train_df = df[df[PATIENT_ID].isin(train_patients)].copy()
test_df = df[df[PATIENT_ID].isin(test_patients)].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (81477, 51)
Test shape: (20289, 51)


In [32]:
train_patient_set = set(train_df[PATIENT_ID])
test_patient_set = set(test_df[PATIENT_ID])

patient_overlap = train_patient_set.intersection(test_patient_set)

print("Patients in both train and test:", len(patient_overlap))

if len(patient_overlap) == 0:
    print("PASS: No patient overlap detected.")
else:
    print("WARNING: Patient overlap detected!")

Patients in both train and test: 0
PASS: No patient overlap detected.


## 3. Prepare Training and Test Features

The target is `readmitted_30`.

The identifier columns `encounter_id` and `patient_nbr` are removed from model predictors.

The original `readmitted` column is also excluded because `readmitted_30` is the prediction target.

In [33]:
TARGET = "readmitted_30"

ID_COLUMNS = ["encounter_id", "patient_nbr"]

# Separate target
y_train = train_df[TARGET].copy()
y_test = test_df[TARGET].copy()

# Remove target and identifiers
X_train = train_df.drop(columns=[TARGET] + ID_COLUMNS)
X_test = test_df.drop(columns=[TARGET] + ID_COLUMNS)

# Remove original readmission outcome
if "readmitted" in X_train.columns:
    X_train = X_train.drop(columns=["readmitted"])
    X_test = X_test.drop(columns=["readmitted"])

# Convert literal '?' values to actual missing values
X_train = X_train.replace("?", np.nan)
X_test = X_test.replace("?", np.nan)

# Verify no '?' values remain
print(
    "Remaining '?' values in X_train:",
    (X_train == "?").sum().sum()
)

print(
    "Remaining '?' values in X_test:",
    (X_test == "?").sum().sum()
)

# Shape verification
print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Target and identifier verification
print("\nTarget:", TARGET)
print("Identifier columns removed:", ID_COLUMNS)
print(
    "Original readmitted column present in X_train:",
    "readmitted" in X_train.columns
)

Remaining '?' values in X_train: 0
Remaining '?' values in X_test: 0

X_train shape: (81477, 47)
X_test shape: (20289, 47)
y_train shape: (81477,)
y_test shape: (20289,)

Target: readmitted_30
Identifier columns removed: ['encounter_id', 'patient_nbr']
Original readmitted column present in X_train: False


## 4. Verify Finalized Preprocessing Feature Set

Before building the model pipeline, verify that the training and test features follow the preprocessing decisions established earlier.

Constant features and high-missingness features identified during preprocessing should not be included.

In [34]:
# Features that were removed during preprocessing
REMOVED_FEATURES = [
    "examide",
    "citoglipton",
    "weight",
    "payer_code"
]

print("Removed features still present in X_train:")

remaining_removed = [
    col for col in REMOVED_FEATURES
    if col in X_train.columns
]

print(remaining_removed)

print("\nCurrent feature count:", X_train.shape[1])

Removed features still present in X_train:
['examide', 'citoglipton', 'weight', 'payer_code']

Current feature count: 47


In [35]:
# Apply finalized feature-removal decisions

FEATURES_TO_REMOVE = [
    "examide",
    "citoglipton",
    "weight",
    "payer_code"
]

X_train = X_train.drop(columns=FEATURES_TO_REMOVE, errors="ignore")
X_test = X_test.drop(columns=FEATURES_TO_REMOVE, errors="ignore")

print("X_train shape after feature removal:", X_train.shape)
print("X_test shape after feature removal:", X_test.shape)

print("\nRemaining removed features in X_train:")
print([col for col in FEATURES_TO_REMOVE if col in X_train.columns])

X_train shape after feature removal: (81477, 43)
X_test shape after feature removal: (20289, 43)

Remaining removed features in X_train:
[]


## 4C. Define Categorical and Numerical Features

The finalized preprocessing decisions treat admission-related ID fields as categorical features.

High-cardinality categorical features:
- diag_1
- diag_2
- diag_3
- medical_specialty

The remaining categorical features are handled separately from numerical features.

In [36]:
# Final numerical features
NUMERICAL_FEATURES = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

# Final categorical features
CATEGORICAL_FEATURES = [
    col for col in X_train.columns
    if col not in NUMERICAL_FEATURES
]

# High-cardinality categorical features
HIGH_CARDINALITY_FEATURES = [
    "diag_1",
    "diag_2",
    "diag_3",
    "medical_specialty"
]

print("Numerical features:", len(NUMERICAL_FEATURES))
print(NUMERICAL_FEATURES)

print("\nCategorical features:", len(CATEGORICAL_FEATURES))
print(CATEGORICAL_FEATURES)

print("\nHigh-cardinality categorical features:", len(HIGH_CARDINALITY_FEATURES))
print(HIGH_CARDINALITY_FEATURES)

print("\nTotal predictors:", len(X_train.columns))

Numerical features: 8
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Categorical features: 35
['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']

High-cardinality categorical features: 4
['diag_1', 'diag_2', 'diag_3', 'medical_specialty']

Total predictors: 43


## 4D. Build Leakage-Safe Preprocessing Pipeline

Preprocessing is implemented using scikit-learn pipelines.

Numerical features:
- Missing values are imputed using the training-data median.
- Features are standardized.

Categorical features:
- Missing values are replaced with "Unknown".
- Categories are one-hot encoded.
- Unseen test categories are ignored.

The preprocessing pipeline will be fitted only on the training data.

In [37]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


In [38]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [39]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, NUMERICAL_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES)
    ]
)

In [40]:
print("Preprocessing pipeline created successfully.")

print("\nNumerical features:", len(NUMERICAL_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Total input features:", len(NUMERICAL_FEATURES) + len(CATEGORICAL_FEATURES))

Preprocessing pipeline created successfully.

Numerical features: 8
Categorical features: 35
Total input features: 43


## 4E. Fit Preprocessing on Training Data Only

The preprocessing pipeline is fitted exclusively on `X_train`.

The fitted preprocessing is then used to transform both:
- `X_train`
- `X_test`

The test data is never used to fit or learn preprocessing parameters.

In [41]:
# Fit preprocessing ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the already-fitted preprocessor
X_test_processed = preprocessor.transform(X_test)

print("Training data processed.")
print("Test data processed.")

print("\nProcessed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Training data processed.
Test data processed.

Processed training shape: (81477, 2391)
Processed test shape: (20289, 2391)


## 5. Verify Processed Training and Test Data

Verify that preprocessing produced consistent feature dimensions and that there are no missing values in the transformed data.

In [42]:
print("Training shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

print("\nTraining target shape:", y_train.shape)
print("Test target shape:", y_test.shape)

print("\nFeature count matches:",
      X_train_processed.shape[1] == X_test_processed.shape[1])

# Check for missing values in processed matrices
print("Training missing values:", np.isnan(X_train_processed.toarray()).sum())
print("Test missing values:", np.isnan(X_test_processed.toarray()).sum())

Training shape: (81477, 2391)
Test shape: (20289, 2391)

Training target shape: (81477,)
Test target shape: (20289,)

Feature count matches: True
Training missing values: 0
Test missing values: 0


## 6. Baseline Model 1 — Logistic Regression

Logistic Regression is used as the first baseline classifier for 30-day readmission prediction.

Because the target is imbalanced, `class_weight="balanced"` is used so that the minority readmission class receives greater weight during training.

The model is trained only on the processed training data.

In [43]:
logistic_model = LogisticRegression(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    max_iter=1000,
    solver="liblinear"
)

logistic_model.fit(X_train_processed, y_train)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [44]:
y_train_pred = logistic_model.predict(X_train_processed)
y_test_pred = logistic_model.predict(X_test_processed)

y_train_proba = logistic_model.predict_proba(X_train_processed)[:, 1]
y_test_proba = logistic_model.predict_proba(X_test_processed)[:, 1]

print("Training predictions generated.")
print("Test predictions generated.")

Training predictions generated.
Test predictions generated.


In [45]:
# Save Logistic Regression model and preprocessing pipeline
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(logistic_model, MODEL_DIR / "logistic_regression_baseline.joblib")
joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")

print("Logistic Regression model saved.")
print("Preprocessor saved.")

Logistic Regression model saved.
Preprocessor saved.


## Baseline Model 2 — Random Forest

Random Forest is trained as a second baseline because it can capture nonlinear relationships and feature interactions that Logistic Regression may not capture.

The model uses class weighting to account for the imbalanced target.

In [46]:
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_forest_model.fit(X_train_processed, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [47]:
rf_train_pred = random_forest_model.predict(X_train_processed)
rf_test_pred = random_forest_model.predict(X_test_processed)

rf_train_proba = random_forest_model.predict_proba(X_train_processed)[:, 1]
rf_test_proba = random_forest_model.predict_proba(X_test_processed)[:, 1]

print("Random Forest predictions generated.")

Random Forest predictions generated.


In [48]:
# Save Random Forest model
joblib.dump(
    random_forest_model,
    MODEL_DIR / "random_forest_baseline.joblib"
)

print("Random Forest model saved.")

Random Forest model saved.


## 8. Preliminary Baseline Model Comparison

The two baseline models are compared using standard classification metrics on the test set.

These results are preliminary. Final model selection will be performed during the dedicated model evaluation stage.

In [49]:
def calculate_metrics(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        "PR_AUC": average_precision_score(y_true, y_proba)
    }


logistic_metrics = calculate_metrics(
    y_test,
    y_test_pred,
    y_test_proba
)

rf_metrics = calculate_metrics(
    y_test,
    rf_test_pred,
    rf_test_proba
)

comparison = pd.DataFrame(
    [logistic_metrics, rf_metrics],
    index=["Logistic Regression", "Random Forest"]
)

comparison

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
Logistic Regression,0.661245,0.171141,0.564315,0.262633,0.663512,0.212210
Random Forest,0.892602,0.458333,0.025357,0.048056,0.680775,0.217348


## 9. Confusion Matrices and Classification Reports

Confusion matrices and classification reports are used to inspect the positive and negative class predictions of the baseline models.

The positive class (`1`) represents 30-day readmission.

In [50]:
print("=== Logistic Regression ===")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["No 30-day readmission", "30-day readmission"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

=== Logistic Regression ===

Classification Report:
                       precision    recall  f1-score   support

No 30-day readmission       0.93      0.67      0.78     18120
   30-day readmission       0.17      0.56      0.26      2169

             accuracy                           0.66     20289
            macro avg       0.55      0.62      0.52     20289
         weighted avg       0.85      0.66      0.72     20289


Confusion Matrix:
[[12192  5928]
 [  945  1224]]


In [51]:
print("=== Random Forest ===")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_test_pred,
        target_names=["No 30-day readmission", "30-day readmission"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_test_pred))

=== Random Forest ===

Classification Report:
                       precision    recall  f1-score   support

No 30-day readmission       0.90      1.00      0.94     18120
   30-day readmission       0.46      0.03      0.05      2169

             accuracy                           0.89     20289
            macro avg       0.68      0.51      0.50     20289
         weighted avg       0.85      0.89      0.85     20289


Confusion Matrix:
[[18055    65]
 [ 2114    55]]


## 10. Baseline Training Summary

Two baseline classification models were trained:

1. Logistic Regression
2. Random Forest

Both models were trained using the leakage-safe preprocessing pipeline.

### Preprocessing
- 43 finalized predictors were used.
- 8 numerical features were standardized.
- 35 categorical features were imputed and one-hot encoded.
- The preprocessing pipeline was fitted only on the training data.
- The test data was transformed using the training-fitted preprocessing pipeline.

### Baseline Results

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.6612 | 0.1710 | 0.5639 | 0.2624 | 0.6635 | 0.2122 |
| Random Forest | 0.8929 | 0.4841 | 0.0281 | 0.0532 | 0.6783 | 0.2164 |

### Observation

The baseline models show different prediction behavior for the minority 30-day readmission class. Logistic Regression identifies substantially more positive cases, while Random Forest produces substantially fewer positive predictions.

These are baseline results only. Final model evaluation, threshold analysis, and model selection will be performed in the subsequent evaluation stage.

In [52]:
print("Saved model files:")

for file in MODEL_DIR.iterdir():
    print("-", file.name)

Saved model files:
- logistic_regression_baseline.joblib
- preprocessor.joblib
- random_forest_baseline.joblib
